# Machine Learning and Spatial Analysis: Demographic Typology and Education–Health Service Adequacy in Uva Province, Sri Lanka

**Final-year research notebook (Google Colab).**

This notebook loads Grama Niladhari (GN)–level census and service-access data, engineers education- and health-relevant indicators, performs exploratory analysis and K-means clustering, flags priority GNs, and exports a master dataset for mapping (e.g. QGIS).

**Age-structure proxies (limitations):** Sri Lankan education stages do not align perfectly with single census bands. Comments in the code document each approximation (e.g. preschool uses `0_4`, which includes ages 0–2).


## 0. Setup

Install packages (Colab may already have some). Set `ROOT` to your data folder: `Path(".")` if you uploaded files to the session, or `/content/drive/MyDrive/...` after mounting Drive.


In [ ]:
# @title 0. Setup — installs, imports, folders
import sys
import subprocess

def pip_install(packages):
    for p in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

try:
    import openpyxl  # noqa: F401
    import sklearn  # noqa: F401
    import geopandas  # noqa: F401
    import pyarrow  # noqa: F401
except ImportError:
    pip_install(["openpyxl", "scikit-learn", "geopandas", "pyarrow"])

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from pathlib import Path
import re
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150

# --- USER: set data root ---
# Option A: files uploaded to Colab session
ROOT = Path(".")
# Option B: Google Drive (uncomment after mounting Drive in the next cell)
# ROOT = Path("/content/drive/MyDrive/YOUR_FOLDER_NAME")

FIG_DIR = Path("figures_eda")
FINAL_DIR = Path("final")
FIG_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT.resolve())
print("Figures ->", FIG_DIR.resolve())
print("Exports ->", FINAL_DIR.resolve())


### Optional: mount Google Drive

Run the cell below only if your data lives on Drive. Otherwise skip it.


In [ ]:
# @title Optional — mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab — skip Drive mount.")


## 1. Load and merge all datasets

Expected files under `ROOT` (adjust names in the config cell if yours differ):

| File | Role |
|------|------|
| `population.xlsx` | GN-level census (backup / join on `gn_number`) |
| `GN_hospital_school_distances_km.csv` | Distances |
| `GN_facility_counts_clean.csv` | School / facility counts |
| `gn_crosswalk_final.csv` | Crosswalk + demographics |
| Shapefiles (optional) | `Badulla District GN Boundary.shp`, `Monaragala District GN Boundary.shp` |

**District fallback:** If shapefiles are missing, the first `N_BADULLA` rows of the distance table are labelled **Badulla** and the remainder **Monaragala** (defaults 573 / 321 — change if your extract differs).


In [ ]:
# @title 1. File names and helpers
POP_XLSX = ROOT / "population.xlsx"
DIST_CSV = ROOT / "GN_hospital_school_distances_km.csv"
FAC_CSV = ROOT / "GN_facility_counts_clean.csv"
XWALK_CSV = ROOT / "gn_crosswalk_final.csv"
SHP_BADULLA = ROOT / "Badulla District GN Boundary.shp"
SHP_MONARAGALA = ROOT / "Monaragala District GN Boundary.shp"

# Row-order district split when shapefiles are unavailable
N_BADULLA = 573
N_MONARAGALA = 321

AGE_COLS = [
    "0_4", "5_9", "10_14", "15_19", "20_24", "25_29", "30_34", "35_39",
    "40_44", "45_49", "50_54", "55_59", "60_64", "65_69", "70_74", "75_79",
    "80_84", "85_89", "90_94", "95_plus",
]

WORK_BANDS = ["15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49", "50_54", "55_59"]
ELDER_BANDS = ["65_69", "70_74", "75_79", "80_84", "85_89", "90_94", "95_plus"]
YOUTH_0_14 = ["0_4", "5_9", "10_14"]
OLD_60_PLUS = ["60_64"] + ELDER_BANDS


def norm_name(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def read_table(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    suf = path.suffix.lower()
    if suf in {".csv"}:
        return pd.read_csv(path, **kwargs)
    if suf in {".xlsx", ".xls"}:
        return pd.read_excel(path, **kwargs)
    raise ValueError(f"Unsupported format: {path}")


def coerce_age_columns(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df


def rename_distance_columns(df):
    m = {}
    for a, b in [
        ("dist_to_nearest_school_km", "dist_school_km"),
        ("dist_to_nearest_hospital_km", "dist_hospital_km"),
    ]:
        if a in df.columns:
            m[a] = b
    return df.rename(columns=m)


print("Configured paths OK.")


In [ ]:
# @title 1. Load base tables
dist = read_table(DIST_CSV)
dist = rename_distance_columns(dist)
if "dist_school_km" not in dist.columns or "dist_hospital_km" not in dist.columns:
    raise ValueError("Distance CSV must contain school/hospital distance columns (see rename_distance_columns).")

fac = read_table(FAC_CSV)
xw = read_table(XWALK_CSV)

# Crosswalk: ensure expected age + id columns exist (flexible column order)
missing_age = [c for c in AGE_COLS if c not in xw.columns]
if missing_age:
    raise ValueError(f"Crosswalk missing age columns: {missing_age[:5]}...")

xw = coerce_age_columns(xw.copy(), AGE_COLS + (["total"] if "total" in xw.columns else []))

pop_xl = None
if POP_XLSX.exists():
    pop_xl = read_table(POP_XLSX)
    if "total" in pop_xl.columns:
        pop_xl["total"] = pd.to_numeric(pop_xl["total"], errors="coerce")
    pop_xl = coerce_age_columns(pop_xl, AGE_COLS)
else:
    print("Note: population.xlsx not found — using crosswalk (+ merges) only.")

fac["name_key"] = fac["Name"].map(norm_name)
xw["name_key"] = xw["Name"].map(norm_name)
dist["name_key"] = dist["Name"].map(norm_name)

print("dist:", dist.shape, "| fac:", fac.shape, "| crosswalk:", xw.shape)


In [ ]:
# @title 1. Shapefiles (optional) → GN codes + district
HAS_SHAPE = SHP_BADULLA.exists() and SHP_MONARAGALA.exists()
gdf_union = None

if HAS_SHAPE:
    import geopandas as gpd
    g_b = gpd.read_file(SHP_BADULLA)
    g_m = gpd.read_file(SHP_MONARAGALA)
    # Try common name fields
    def pick_name_col(g):
        for c in ["Name", "GN_Name", "GN_NAME", "name", "NAME", "ADM4_EN"]:
            if c in g.columns:
                return c
        return None

    nb = pick_name_col(g_b)
    nm = pick_name_col(g_m)
    if nb is None or nm is None:
        print("Warning: could not detect name column in shapefiles; falling back to CSV row-order district.")
        HAS_SHAPE = False
    else:
        g_b = g_b.copy()
        g_m = g_m.copy()
        g_b["district"] = "Badulla"
        g_m["district"] = "Monaragala"
        g_b["bnd_name"] = g_b[nb].astype(str)
        g_m["bnd_name"] = g_m[nm].astype(str)
        g_b["name_key"] = g_b["bnd_name"].map(norm_name)
        g_m["name_key"] = g_m["bnd_name"].map(norm_name)
        # GN code from common fields
        def pick_gn_code_col(g):
            for c in ["GN_code", "GN_CODE", "GN_NO", "D_CODE", "code", "OBJECTID"]:
                if c in g.columns:
                    return c
            return None
        cb, cm = pick_gn_code_col(g_b), pick_gn_code_col(g_m)
        if cb:
            g_b["GN_code_shp"] = g_b[cb]
        else:
            g_b["GN_code_shp"] = np.arange(len(g_b))
        if cm:
            g_m["GN_code_shp"] = g_m[cm]
        else:
            g_m["GN_code_shp"] = np.arange(len(g_m))
        gdf_union = pd.concat([g_b, g_m], ignore_index=True)
        print("Shapefiles loaded:", g_b.shape, g_m.shape)
else:
    print("Shapefiles not found — using distance CSV + row-order district fallback.")


In [ ]:
# @title 1. Build master spine from distance CSV
master = dist.copy()
master = master.reset_index(drop=True)

if "GN_code" not in master.columns:
    master["GN_code"] = master.index

if HAS_SHAPE and gdf_union is not None:
    # Spatial join is heavy; match by normalized name + district when possible
    m_b = master.iloc[:N_BADULLA].copy()
    m_r = master.iloc[N_BADULLA:].copy()
    m_b["district"] = "Badulla"
    m_r["district"] = "Monaragala"
    master = pd.concat([m_b, m_r], ignore_index=True)
    shp_sub = gdf_union[["district", "name_key", "GN_code_shp"]].drop_duplicates(subset=["district", "name_key"])
    master = master.merge(shp_sub, on=["district", "name_key"], how="left")
    master["GN_code"] = master["GN_code_shp"].fillna(master["GN_code"])
    master.drop(columns=[c for c in ["GN_code_shp"] if c in master.columns], inplace=True)
else:
    master["district"] = np.where(master.index < N_BADULLA, "Badulla", "Monaragala")

# Merge facility counts (case-insensitive name)
master = master.merge(
    fac[["name_key", "Edu_Count", "Health_Count"]],
    on="name_key",
    how="left",
)
master["Edu_Count"] = master["Edu_Count"].fillna(0)
master["Health_Count"] = master["Health_Count"].fillna(0)

print("After facility merge:", master.shape)


In [ ]:
# @title 1. Merge crosswalk (exact then fuzzy within district)

def merge_crosswalk_exact(left, xwalk):
    key_cols = ["district", "name_key"]
    x = xwalk.copy()
    # Avoid duplicate column names on merge
    demo_cols = ["gn_number", "area_name", "total", "match_status"] + AGE_COLS
    demo_cols = [c for c in demo_cols if c in x.columns]
    x = x[key_cols + demo_cols].drop_duplicates(subset=key_cols)
    out = left.merge(x, on=key_cols, how="left", suffixes=("", "_xw"))
    return out


def fuzzy_fill_row(name_key, district, xwalk, threshold=0.82):
    if not name_key:
        return None
    sub = xwalk[xwalk["district"] == district]
    choices = sub["name_key"].unique().tolist()
    if not choices:
        return None
    match = difflib.get_close_matches(name_key, choices, n=1, cutoff=threshold)
    if not match:
        return None
    hit = sub[sub["name_key"] == match[0]].iloc[0]
    return hit


master = merge_crosswalk_exact(master, xw)

miss = master["total"].isna() if "total" in master.columns else pd.Series(True, index=master.index)
# If total missing, try fuzzy
fill_demo = ["gn_number", "area_name", "total", "match_status"] + AGE_COLS
fill_demo = [c for c in fill_demo if c in xw.columns]

for i in master.index[miss.fillna(True)]:
    hk = master.at[i, "name_key"]
    dist_ = master.at[i, "district"]
    hit = fuzzy_fill_row(hk, dist_, xw)
    if hit is None:
        continue
    for c in fill_demo:
        master.at[i, c] = hit[c]

# Supplement from population.xlsx on gn_number if still missing totals
if pop_xl is not None and "gn_number" in master.columns and "gn_number" in pop_xl.columns:
    pop = pop_xl.copy()
    demo_from_pop = ["total"] + AGE_COLS
    demo_from_pop = [c for c in demo_from_pop if c in pop.columns]
    pop_sub = pop[["gn_number"] + demo_from_pop].drop_duplicates(subset=["gn_number"])
    pop_sub = pop_sub.assign(_gn_join=pop_sub["gn_number"].astype(str))
    m2 = master.copy()
    m2["_gn_join"] = m2["gn_number"].apply(lambda x: str(x).strip() if pd.notna(x) else np.nan)
    m2 = m2.merge(pop_sub.drop(columns=["gn_number"]), on="_gn_join", how="left", suffixes=("", "_pop"))
    m2.drop(columns=["_gn_join"], inplace=True)
    for c in demo_from_pop:
        pc = f"{c}_pop"
        if pc not in m2.columns:
            continue
        if c in m2.columns:
            m2[c] = m2[c].fillna(m2[pc])
        else:
            m2[c] = m2[pc]
        m2.drop(columns=[pc], inplace=True)
    master = m2

# Coerce numeric
master = coerce_age_columns(master, AGE_COLS + ["total"])

print("After crosswalk (+ optional population) merge:", master.shape)
print("Rows with total population:", master["total"].notna().sum())


## 2. Feature engineering (education & health)

**Education (Sri Lanka stage proxies):**

| Stage | Census | Variable | Limitation |
|-------|--------|----------|------------|
| Preschool (ECCD proxy) | `0_4` | `preschool` | Includes 0–2; cannot isolate 3–4. |
| Primary | `5_9` | `primary` | Aligned. |
| Junior secondary proxy | `10_14` | `junior_secondary` | Age 14 is partly senior secondary. |
| Senior secondary + collegiate | `15_19` | `senior_secondary_plus` | O/L vs A/L not separable. |
| School-age total | sum 0–19 bands | `total_school_age` | Ages 0–19. |

**Health:** elderly share (65+), broad dependency ratio (0–14 and 60+ vs working 15–59), distances and facility counts retained.


In [ ]:
# @title 2. Feature engineering
df = master.copy()

# Totals from bands if total missing
band_sum = df[AGE_COLS].sum(axis=1)
df["total"] = df["total"].fillna(band_sum)

# Education structure (see markdown limitations)
df["preschool"] = df["0_4"]
df["primary"] = df["5_9"]
df["junior_secondary"] = df["10_14"]
df["senior_secondary_plus"] = df["15_19"]
df["total_school_age"] = df["preschool"] + df["primary"] + df["junior_secondary"] + df["senior_secondary_plus"]

t = df["total"].replace(0, np.nan)
for col, pcol in [
    ("preschool", "p_preschool"),
    ("primary", "p_primary"),
    ("junior_secondary", "p_junior_sec"),
    ("senior_secondary_plus", "p_senior_plus"),
    ("total_school_age", "p_school_age"),
]:
    df[pcol] = (df[col] / t * 100).replace([np.inf, -np.inf], np.nan)

# Working age 15–59 and education dependency ratio
df["working_age_15_59"] = df[WORK_BANDS].sum(axis=1)
w = df["working_age_15_59"].replace(0, np.nan)
df["education_dependency_ratio"] = (df["total_school_age"] / w * 100).replace([np.inf, -np.inf], np.nan)

# Health / broad dependency
df["elderly_65plus"] = df[ELDER_BANDS].sum(axis=1)
df["p_elderly"] = (df["elderly_65plus"] / t * 100).replace([np.inf, -np.inf], np.nan)

young = df[YOUTH_0_14].sum(axis=1)
old_broad = df[OLD_60_PLUS].sum(axis=1)
df["dependency_ratio"] = ((young + old_broad) / w * 100).replace([np.inf, -np.inf], np.nan)

# Ensure distance / count columns present
for c in ["dist_school_km", "dist_hospital_km", "Edu_Count", "Health_Count"]:
    if c not in df.columns:
        print("Warning: missing column", c)

master = df
print("Feature engineering complete. Columns:", len(master.columns))


## 3. Data quality check

Duplicate `GN_code` values, missingness, and IQR-based outlier counts for key numeric fields.


In [ ]:
# @title 3. Data quality
key_numeric = [
    "total", "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "dependency_ratio",
    "dist_school_km", "dist_hospital_km", "Edu_Count", "Health_Count",
]
key_numeric = [c for c in key_numeric if c in master.columns]

print("Shape:", master.shape)
print("\nMissing % (top 15):")
miss = (master.isna().mean() * 100).sort_values(ascending=False)
print(miss.head(15).round(2).to_string())

if "GN_code" in master.columns:
    dup = master["GN_code"].duplicated().sum()
    print(f"\nDuplicate GN_code rows: {dup}")
else:
    print("\nNo GN_code column for duplicate check.")

def iqr_outlier_counts(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) < 10:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0, q1, q3
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), lo, hi

print("\nIQR outlier counts:")
for c in key_numeric:
    cnt, lo, hi = iqr_outlier_counts(master[c])
    print(f"  {c}: {cnt} outliers (approx bounds {lo:.3g} … {hi:.3g})")


## 4. Descriptive statistics

Summary for all key analysis variables (mean, SD, quartiles, min, max).


In [ ]:
# @title 4. Descriptive statistics
desc = master[key_numeric].describe(percentiles=[0.25, 0.5, 0.75]).T
desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
display(desc.round(4))


## 5. GN-level distribution analysis

Histograms + KDE, boxplots, and violin plots by district. Figures saved under `figures_eda/`.


In [ ]:
# @title 5. Histograms + KDE
hist_vars = [
    "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "dist_school_km", "dist_hospital_km",
]
hist_vars = [v for v in hist_vars if v in master.columns]

for v in hist_vars:
    plt.figure(figsize=(7, 4))
    s = pd.to_numeric(master[v], errors="coerce").dropna()
    sns.histplot(s, kde=True, color="steelblue", edgecolor="white")
    plt.title(f"Distribution: {v}")
    plt.xlabel(v)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"hist_kde_{v}.png")
    plt.show()
    plt.close()
plt.close("all")


In [ ]:
# @title 5. Boxplots and violin plots by district
if "district" in master.columns:
    for v in hist_vars:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=master, x="district", y=v, palette="Set2")
        plt.title(f"Boxplot by district: {v}")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"box_{v}_district.png")
        plt.show()
        plt.close()

        plt.figure(figsize=(7, 4))
        sns.violinplot(data=master, x="district", y=v, palette="Pastel1", cut=0, inner="quartile")
        plt.title(f"Violin by district: {v}")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"violin_{v}_district.png")
        plt.show()
        plt.close()
    plt.close("all")
else:
    print("No district column — skipping stratified box/violin plots.")


## 6. Top / bottom GN analysis

Tables and horizontal bar charts for extremes — useful for slides.


In [ ]:
# @title 6. Top/bottom tables + bar charts

def show_top_bottom(var, n=10, higher_is_worse_for_distance=True):
    d = master[["GN_code", "Name", "district", var]].dropna(subset=[var]).copy()
    d = d.sort_values(var, ascending=False)
    top = d.head(n)
    bottom = d.tail(n)
    print(f"\n=== Top {n} by {var} ===")
    display(top.reset_index(drop=True))
    print(f"\n=== Bottom {n} by {var} ===")
    display(bottom.reset_index(drop=True))

    plt.figure(figsize=(8, 4))
    sns.barplot(data=top, y="Name", x=var, color="coral", orient="h")
    plt.title(f"Top {n} GNs — {var}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"bar_top_{var}.png")
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 4))
    sns.barplot(data=bottom.sort_values(var), y="Name", x=var, color="seagreen", orient="h")
    plt.title(f"Bottom {n} GNs — {var}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"bar_bottom_{var}.png")
    plt.show()
    plt.close()


top_vars = [
    "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "dist_school_km", "dist_hospital_km", "p_elderly",
]
top_vars = [v for v in top_vars if v in master.columns]

for v in top_vars:
    show_top_bottom(v, n=10)

# Bottom distances = best served (closest facilities)
for v in ["dist_school_km", "dist_hospital_km"]:
    if v not in master.columns:
        continue
    d = master[["GN_code", "Name", "district", v]].dropna(subset=[v]).copy()
    best = d.nsmallest(10, v)
    print(f"\n=== Best served (smallest {v}) — bottom 10 distances ===")
    display(best.reset_index(drop=True))
    plt.figure(figsize=(8, 4))
    sns.barplot(data=best, y="Name", x=v, color="teal", orient="h")
    plt.title(f"Closest 10 GNs — {v}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"bar_best_served_{v}.png")
    plt.show()
    plt.close()
plt.close("all")


## 7. Relationship analysis

Correlation heatmap, selected scatter plots, and a pairplot of education-stage percentages.


In [ ]:
# @title 7. Correlation heatmap
corr_cols = [
    "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "dependency_ratio", "p_elderly",
    "dist_school_km", "dist_hospital_km", "Edu_Count", "Health_Count", "total",
]
corr_cols = [c for c in corr_cols if c in master.columns]
cm = master[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=False, cmap="RdBu_r", center=0, linewidths=0.5)
plt.title("Correlation matrix — education, health, access")
plt.tight_layout()
plt.savefig(FIG_DIR / "corr_heatmap.png")
plt.show()
plt.close()


In [ ]:
# @title 7. Scatter plots
pairs = [
    ("p_school_age", "dist_school_km"),
    ("p_elderly", "dist_hospital_km"),
    ("education_dependency_ratio", "total"),
    ("total", "dist_school_km"),
]
for x, y in pairs:
    if x not in master.columns or y not in master.columns:
        continue
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=master, x=x, y=y, hue="district" if "district" in master.columns else None, alpha=0.7)
    plt.title(f"{y} vs {x}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"scatter_{x}__{y}.png")
    plt.show()
    plt.close()
plt.close("all")


In [ ]:
# @title 7. Pairplot (education-stage %)
pair_cols = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus"] if c in master.columns]
if len(pair_cols) >= 2:
    pp = sns.pairplot(master[pair_cols].dropna(), corner=True, plot_kws={"s": 18, "alpha": 0.6})
    pp.fig.suptitle("Pairplot — education-stage population shares (%)", y=1.02)
    plt.savefig(FIG_DIR / "pairplot_education_stages.png")
    plt.show()
    plt.close("all")
else:
    print("Insufficient columns for pairplot.")


## 8. Priority GN flags (hotspots)

- **Education:** high school-age share and far from nearest school (both above 75th percentile).
- **Health:** high elderly share and far from nearest hospital.
- **Dual:** either education or health priority.


In [ ]:
# @title 8. Priority flags
m = master.copy()

def flag_high(field, p=0.75):
    return m[field] >= m[field].quantile(p)

m["priority_education"] = flag_high("p_school_age") & flag_high("dist_school_km")
m["priority_health"] = flag_high("p_elderly") & flag_high("dist_hospital_km")
m["priority_dual"] = m["priority_education"] | m["priority_health"]

flag_cols = [
    "GN_code", "Name", "district",
    "p_school_age", "dist_school_km", "p_elderly", "dist_hospital_km",
    "priority_education", "priority_health", "priority_dual",
    "Edu_Count", "Health_Count",
]
flag_cols = [c for c in flag_cols if c in m.columns]

flagged = m[m["priority_dual"]][flag_cols].sort_values(
    by=["priority_education", "priority_health", "p_school_age", "p_elderly"],
    ascending=False,
)
print("Flagged GN count:", len(flagged))
display(flagged)

master = m


## 9. K-means clustering (education typology)

Features: preschool–senior+ shares, `log_total`, and broad `dependency_ratio`. Standardized; silhouette scores for *k* = 2…10; best *k* retained. Cluster profiles and heatmap of cluster means.


In [ ]:
# @title 9. K-means — silhouette, fit, profiles
feat_cols = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus"] if c in master.columns]
if "total" not in master.columns or "dependency_ratio" not in master.columns:
    raise ValueError("Need total and dependency_ratio for clustering.")

Xraw = master[feat_cols + ["total", "dependency_ratio"]].copy()
Xraw["log_total"] = np.log1p(pd.to_numeric(Xraw["total"], errors="coerce"))
Xraw = Xraw.drop(columns=["total"])
X = Xraw.dropna()
idx = X.index

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

sil = {}
best_k, best_score = None, -1.0
for k in range(2, 11):
    if len(Xs) <= k:
        continue
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(Xs)
    try:
        score = silhouette_score(Xs, labels)
    except Exception:
        continue
    sil[k] = score
    if score > best_score:
        best_k, best_score = k, score

if best_k is None:
    best_k = 3

print("Silhouette by k:", sil)
print("Chosen k:", best_k, "score:", round(best_score, 4))

km_final = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
cluster_labels = pd.Series(np.nan, index=master.index)
cluster_labels.loc[idx] = km_final.fit_predict(Xs)
master["cluster_kmeans"] = cluster_labels

# Profiles
prof_cols = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "total"] if c in master.columns]
profiles = master.dropna(subset=["cluster_kmeans"]).groupby("cluster_kmeans")[prof_cols].mean()
display(profiles.round(3))

plt.figure(figsize=(max(8, best_k + 2), 4))
sns.heatmap(profiles.T, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Cluster profiles — mean education % and total pop")
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_cluster_means_heatmap.png")
plt.show()
plt.close()


## 10. Final dataset export

CSV for QGIS / Excel; Parquet when `pyarrow` is available (installed in setup).


In [ ]:
# @title 10. Export master dataset
export_df = master.copy()
# Drop geometry if present
if hasattr(export_df, "geometry"):
    try:
        export_df = export_df.drop(columns=["geometry"])
    except Exception:
        pass
for col in list(export_df.columns):
    if str(export_df[col].dtype) == "geometry":
        export_df = export_df.drop(columns=[col])

csv_path = FINAL_DIR / "UVA_master_dataset_education_health.csv"
export_df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

try:
    pq_path = FINAL_DIR / "UVA_master_dataset_education_health.parquet"
    export_df.to_parquet(pq_path, index=False)
    print("Saved:", pq_path)
except Exception as e:
    print("Parquet export skipped:", e)


## 11. Auto-generated insights (draft bullets)

These lines summarise patterns in **this run** — edit wording for your thesis after checking maps and robustness.


In [ ]:
# @title 11. Presentation-ready bullet points
m = master
bullets = []

def qstr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return "n/a"
    return f"p25={s.quantile(0.25):.2f}, median={s.median():.2f}, p75={s.quantile(0.75):.2f}"

bullets.append(
    f"Coverage: {len(m)} GNs in the master table; "
    f"{m['total'].notna().sum()} with non-missing total population."
)
bullets.append(
    f"School-age share (% of population): {qstr(m['p_school_age'])} — wide GN-level spread supports typology analysis."
)
bullets.append(
    f"Education dependency ratio: {qstr(m['education_dependency_ratio'])} — reflects children/youth relative to working-age (15–59) population."
)
bullets.append(
    f"Elderly share: {qstr(m['p_elderly'])} — relevant for hospital-access pressure alongside distance."
)
bullets.append(
    f"Distance to nearest school (km): {qstr(m['dist_school_km'])}; hospital: {qstr(m['dist_hospital_km'])}."
)
bullets.append(
    f"Priority GNs (education): {int(m['priority_education'].sum())}; "
    f"(health): {int(m['priority_health'].sum())}; "
    f"either: {int(m['priority_dual'].sum())}."
)
if "cluster_kmeans" in m.columns and m["cluster_kmeans"].notna().any():
    nk = int(m["cluster_kmeans"].dropna().nunique())
    bullets.append(
        f"K-means (k={nk}) separates GNs by stage-mix and scale — see cluster mean heatmap for interpretation."
    )

for b in bullets:
    print("•", b)


## 12. QGIS: join exported CSV to GN boundaries

1. **Add vector layer:** Layer → Add Layer → Add Vector Layer — load `Badulla District GN Boundary.shp` and `Monaragala District GN Boundary.shp` (or a merged layer).
2. **Import CSV:** Layer → Add Layer → Add Delimited Text Layer — choose `UVA_master_dataset_education_health.csv`. Select the **X** and **Y** fields only if your CSV has coordinates (this export does not, by default).
3. **Join:** Open the boundary layer’s **Properties** → **Joins** → **+**.  
   - Join layer = your CSV.  
   - Join field = the column that matches the shapefile (often `Name`, `GN_code`, or a district-specific ID — inspect both attribute tables).  
   - Target field = the corresponding field in the shapefile.  
4. **Symbology:** Properties → Symbology → Graduated — choose `p_school_age`, `dist_hospital_km`, `cluster_kmeans`, or `priority_dual` for choropleth maps.
5. **Export:** Right-click the joined layer → Export → Save Features As… to bake the join into a new file.

If names differ slightly between CSV and shapefile, build a small crosswalk key in Excel or use QGIS **Refactor fields** / **Join by location** after standardising names.

---

**Reproducibility note:** Census band proxies for education stages are documented in section 2. Interpret coefficients and cluster labels with these limitations in your discussion section.
